# 第4章  向量的加减与数乘 —— 平移与缩放

> 本章目标：建立向量运算的几何直觉——加法=平移，数乘=缩放。用quiver画向量，理解词嵌入语义运算和Agent状态迁移。
> 前置知识：第3章（向量、NumPy数组、shape）


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print('环境就绪')


---
## 4.1  向量加法 —— 两次走路的合成

向东3公里+向北4公里=合成5公里。向量加法：a+b。
平行四边形法则。残差连接x+Sublayer(x)就是向量加法。

AI连接：第29章Transformer残差连接。


In [ ]:
import numpy as np
a = np.array([3.0, 0.0]); b = np.array([0.0, 4.0])
c = a + b
print(f'a={a}')
print(f'b={b}')
print(f'a+b={c}, distance={np.sqrt(c[0]**2+c[1]**2):.1f}')
print(f'commutative: {(a+b==b+a).all()}')
print(f'a+0={a+np.zeros(2)} (zero vector)')


---
## 4.2  向量数乘 —— 不转弯，只伸缩

c*v=[c*v1,c*v2,...]。c>1拉长，0<c<1缩短。方向不变。
lr*gradient就是数乘——梯度定方向，学习率定步长。

AI连接：lr*grad贯穿Ch12到Ch35。


In [ ]:
import numpy as np
v=np.array([3.0,4.0]); vl=np.sqrt(np.sum(v**2))
print(f'v={v}, length={vl:.1f}')
for c in[2.,.5,-1.,0.]:
  sv=c*v; sl=np.sqrt(np.sum(sv**2))
  print(f'  c={c:4.1f}: {c}*v={sv}, len={sl:.1f}')
grad=np.array([.5,-.3])
print(f'lr=0.01: {0.01*grad} (small)  lr=1.0: {1.0*grad} (large)')


---
## 4.3  几何可视化 —— quiver画向量

quiver(起点X,起点Y,方向X,方向Y)。左：平行四边形(v+u)。右：数乘(2v)。

AI连接：quiver用于Ch12梯度场，Ch17 PCA方向。


In [ ]:
import numpy as np, matplotlib.pyplot as plt
v=np.array([3.,1.]); u=np.array([1.,2.]); w=v+u; v2=2.*v
fig,axes=plt.subplots(1,2,figsize=(12,5)); o=np.array([0.,0.])
axes[0].quiver(*o,*v,angles='xy',scale_units='xy',scale=1,color='blue',width=.015,label='v')
axes[0].quiver(*o,*u,angles='xy',scale_units='xy',scale=1,color='green',width=.015,label='u')
axes[0].quiver(*v,*u,angles='xy',scale_units='xy',scale=1,color='green',width=.012,alpha=.5,ls='--')
axes[0].quiver(*u,*v,angles='xy',scale_units='xy',scale=1,color='blue',width=.012,alpha=.5,ls='--')
axes[0].quiver(*o,*w,angles='xy',scale_units='xy',scale=1,color='red',width=.018,label='v+u')
axes[0].set_xlim(-1,5);axes[0].set_ylim(-1,4);axes[0].set_title('Parallelogram')
axes[1].quiver(*o,*v,angles='xy',scale_units='xy',scale=1,color='blue',width=.015,label='v')
axes[1].quiver(*o,*v2,angles='xy',scale_units='xy',scale=1,color='red',width=.015,label='2v')
axes[1].set_xlim(-1,7);axes[1].set_ylim(-1,3);axes[1].set_title('Scalar: 2v')
plt.tight_layout();plt.show()
print('Left: red=v+u diagonal  Right: 2v same direction')


---
## 4.4  AI应用：词嵌入语义运算

2013 word2vec: v(国王)-v(男人)+v(女人) ~ v(女王)。
向量减法去属性，加法加属性。方向编码语义。

AI连接：Ch5余弦相似度，Ch29 Q·K注意力。


In [ ]:
import numpy as np
wv={'国王':np.array([.9,.7,.5,.1]),'男人':np.array([.6,.7,.2,.1]),'女王':np.array([.4,.2,.6,.0]),'女人':np.array([.1,.2,.3,.0])}
def cs(a,b): return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))
r=wv['国王']-wv['男人']+wv['女人']
print('king-man+woman=')
for w,v in wv.items():
  s=cs(r,v); m=' <-- best!' if w=='女王' else ''
  print(f'  sim with {w}: {s:.4f}{m}')


---
## 4.5  Agent规划：状态向量与动作向量的加法

ReAct: state_new = state_current + action。
状态迁移=语义空间中向量平移。Agent行为=平移链。

AI连接：Ch5.5 RAG, Ch6.6 Continuous Batching。


In [ ]:
import numpy as np, matplotlib.pyplot as plt
np.random.seed(42)
st=np.array([0.,0.])
acts=[np.array([2.,1.]),np.array([-1.,2.5]),np.array([1.5,-1.]),np.array([.5,2.])]
states=[st.copy()]
for a in acts: st=st+a; states.append(st.copy())
states=np.array(states)
fig,ax=plt.subplots(figsize=(8,7))
cols=plt.cm.viridis(np.linspace(0,1,len(acts)))
for i,(sb,a) in enumerate(zip(states[:-1],acts)):
  ax.quiver(sb[0],sb[1],a[0],a[1],angles='xy',scale_units='xy',scale=1,color=cols[i],width=.03,alpha=.8)
ax.plot(states[:,0],states[:,1],'ko-',ms=8,lw=1.5,label='Path')
ax.plot(states[0,0],states[0,1],'go',ms=12,label='Start')
ax.plot(states[-1,0],states[-1,1],'r*',ms=18,label='End')
for i,(sb,a) in enumerate(zip(states[:-1],acts)):
  mid=sb+a/2
  ax.annotate(['Search','Analyze','Synth','Reply'][i],xy=(mid[0],mid[1]),fontsize=9,ha='center',color=cols[i],fontweight='bold')
ax.set_xlim(-.5,4.5);ax.set_ylim(-.5,5.5);ax.set_title('Agent State: state+=action');ax.legend();plt.show()
print(f'Start->End: {states[0]}->{states[-1].round(2)}')
print(f'Sum actions={sum(acts)}, End-Start={states[-1]-states[0]}')


---
## 习题

> 在下方新建代码单元格作答。

1. （概念）用走路比喻解释向量加法的平行四边形法则。
2. （概念）lr在参数更新中执行了什么向量运算？lr=0和lr极大各会怎样？
3. （代码）画a=[2,3]/b=[1,-1]/a+b/a-b/2a五箭头quiver图。
4. （代码）5个随机2D动作模拟Agent状态迁移,quiver画轨迹。
5. （代码）6维词向量验证中国-北京+巴黎~法国。


---

> 章末钩子：两个向量间还有更深层的互动——它们指向的方向有多一致？用点积量化。
> 预览：下一章——**向量的点积（内积）**。

> 提示：完成后运行 Kernel -> Restart & Run All 验证。
